In [22]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PooledOLS, PanelOLS
import warnings
warnings.filterwarnings('ignore')

# ── Load and prepare panel structure ─────────────────────────────────────────
df = pd.read_csv('state_labor_panel_data.csv', parse_dates=['Date'])

# linearmodels requires a MultiIndex of (entity, time)
df = df.set_index(['State', 'Date'])

# Define variables used across all three specifications
Y       = df['ln_Wage']
PREDS   = ['Hiring_Rate', 'Friction_Proxy']          # primary predictors
CONTROLS= ['Unemployment_Rate', 'LH_Share']           # control variables
X_VARS  = PREDS + CONTROLS

print(f"Panel dimensions: {df.index.get_level_values('State').nunique()} states "
      f"x {df.index.get_level_values('Date').nunique()} months "
      f"= {len(df):,} observations")
display(df[['ln_Wage'] + X_VARS].describe().round(4))

Panel dimensions: 25 states x 108 months = 2,700 observations


,ln_Wage,Hiring_Rate,Friction_Proxy,Unemployment_Rate,LH_Share
count,2700.0000,2700.0000,2700.0000,2700.0000,2700.0000
mean,3.3440,0.0357,0.0118,4.6929,0.0008
std,0.1406,0.0067,0.0122,2.0784,0.0000
min,3.0277,0.0170,-0.0446,1.9000,0.0006
25%,3.2363,0.0311,0.0033,3.5000,0.0008
50%,3.3420,0.0351,0.0095,4.2000,0.0008
75%,3.4383,0.0394,0.0200,5.1000,0.0008
max,3.7084,0.0901,0.0651,30.5000,0.0009


In [23]:
# ── Specification 1: Pooled OLS ───────────────────────────────────────────────
# Treats all 25 states x 108 months as one flat dataset.
# No grouping by state — purely establishes the raw correlation.
# Standard errors clustered by state to account for within-state serial correlation.

print("=" * 65)
print("  SPECIFICATION 1 — Pooled OLS (Baseline)")
print("  ln(Wage) = B0 + B1*HiringRate + B2*FrictionProxy + Controls")
print("=" * 65)

X1 = sm.add_constant(df[X_VARS])

spec1 = PooledOLS(Y, X1).fit(
    cov_type='clustered',
    cluster_entity=True    # clusters SEs by state
)

print(spec1.summary)

  SPECIFICATION 1 — Pooled OLS (Baseline)
  ln(Wage) = B0 + B1*HiringRate + B2*FrictionProxy + Controls
                          PooledOLS Estimation Summary                          
Dep. Variable:                ln_Wage   R-squared:                        0.3609
Estimator:                  PooledOLS   R-squared (Between):              0.3484
No. Observations:                2700   R-squared (Within):               0.3760
Date:                Mon, Apr 27 2026   R-squared (Overall):              0.3609
Time:                        13:58:48   Log-likelihood                    2070.1
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      380.46
Entities:                          25   P-value                           0.0000
Avg Obs:                       108.00   Distribution:                  F(4,2695)
Min Obs:                       108.00                                           
Max O

In [24]:
# ── Specification 2: State (Entity) Fixed Effects ────────────────────────────
# Absorbs all time-invariant differences between states —
# e.g., permanent cost-of-living gaps between NY and TN never change,
# so the state FE soaks them up, leaving only within-state variation over time.
# This is why we drop the constant — the entity dummies replace it.

print("=" * 65)
print("  SPECIFICATION 2 — State Fixed Effects")
print("  ln(Wage) = B1*HiringRate + B2*FrictionProxy + Controls")
print("           + State Fixed Effect")
print("=" * 65)

X2 = df[X_VARS]   # no constant — absorbed by entity effects

spec2 = PanelOLS(
    Y, X2,
    entity_effects=True,
    time_effects=False
).fit(
    cov_type='clustered',
    cluster_entity=True
)

print(spec2.summary)

  SPECIFICATION 2 — State Fixed Effects
  ln(Wage) = B1*HiringRate + B2*FrictionProxy + Controls
           + State Fixed Effect
                          PanelOLS Estimation Summary                           
Dep. Variable:                ln_Wage   R-squared:                        0.5793
Estimator:                   PanelOLS   R-squared (Between):              0.2145
No. Observations:                2700   R-squared (Within):               0.5793
Date:                Mon, Apr 27 2026   R-squared (Overall):              0.2148
Time:                        13:58:58   Log-likelihood                    3704.6
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      919.53
Entities:                          25   P-value                           0.0000
Avg Obs:                       108.00   Distribution:                  F(4,2671)
Min Obs:                       108.00                        

In [25]:
# ── Specification 3: Two-Way Fixed Effects ────────────────────────────────────
# Our preferred specification. Adds time fixed effects on top of state FEs.
# Time FEs absorb any shock that hit ALL states in the same month equally —
# e.g., COVID (Apr 2020), Fed rate hikes (2022), national recessions.
# What's left after both sets of dummies is the clean within-state,
# within-month variation we need to identify B2 (Friction Proxy).

print("=" * 65)
print("  SPECIFICATION 3 — Two-Way Fixed Effects (Preferred)")
print("  ln(Wage) = B1*HiringRate + B2*FrictionProxy + Controls")
print("           + State FE + Time FE")
print("=" * 65)

X3 = df[X_VARS]   # no constant — absorbed by both sets of effects

spec3 = PanelOLS(
    Y, X3,
    entity_effects=True,
    time_effects=True      # <-- this is the only change from spec 2
).fit(
    cov_type='clustered',
    cluster_entity=True
)

print(spec3.summary)

  SPECIFICATION 3 — Two-Way Fixed Effects (Preferred)
  ln(Wage) = B1*HiringRate + B2*FrictionProxy + Controls
           + State FE + Time FE
                          PanelOLS Estimation Summary                           
Dep. Variable:                ln_Wage   R-squared:                        0.0284
Estimator:                   PanelOLS   R-squared (Between):              0.0359
No. Observations:                2700   R-squared (Within):               0.0579
Date:                Mon, Apr 27 2026   R-squared (Overall):              0.0359
Time:                        13:59:09   Log-likelihood                    7253.8
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      18.757
Entities:                          25   P-value                           0.0000
Avg Obs:                       108.00   Distribution:                  F(4,2564)
Min Obs:                       108.00          

In [27]:
# ── Side-by-side coefficient comparison across all three specifications ────────
# This is the standard way to present sequential panel models in a paper.
# You want to see if B2 (Friction Proxy) stays negative and significant
# as you add more controls — that's what makes the result credible.

rows = []
for var in X_VARS:
    row = {'Variable': var}
    for label, res in [('Pooled OLS', spec1),
                       ('State FE',   spec2),
                       ('TWFE',       spec3)]:
        coef = res.params[var]
        pval = res.pvalues[var]
        stars = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
        row[label] = f"{coef:.4f}{stars}"
    rows.append(row)

# Add model fit rows
for label, res in [('Pooled OLS', spec1), ('State FE', spec2), ('TWFE', spec3)]:
    pass

comparison = pd.DataFrame(rows).set_index('Variable')

# Append R-squared and N
fit_rows = pd.DataFrame([
    {'Variable': 'R² (within)',
     'Pooled OLS': f"{spec1.rsquared:.4f}",
     'State FE':   f"{spec2.rsquared:.4f}",
     'TWFE':       f"{spec3.rsquared:.4f}"},
    {'Variable': 'N',
     'Pooled OLS': f"{spec1.nobs:,.0f}",
     'State FE':   f"{spec2.nobs:,.0f}",
     'TWFE':       f"{spec3.nobs:,.0f}"},
    {'Variable': 'State FE',
     'Pooled OLS': 'No', 'State FE': 'Yes', 'TWFE': 'Yes'},
    {'Variable': 'Time FE',
     'Pooled OLS': 'No', 'State FE': 'No',  'TWFE': 'Yes'},
]).set_index('Variable')

comparison = pd.concat([comparison, fit_rows])

print("\n  *** p<0.01  ** p<0.05  * p<0.10")
print("  Standard errors clustered by state\n")
display(comparison)


  *** p<0.01  ** p<0.05  * p<0.10
  Standard errors clustered by state



,Pooled OLS,State FE,TWFE
Variable,,,
Hiring_Rate,-3.4620**,4.7035***,0.1722
Friction_Proxy,6.6328***,6.1727***,0.3967*
Unemployment_Rate,0.0181***,0.0096***,0.0034*
LH_Share,515.5115,121.3192,43.3656
R² (within),0.3609,0.5793,0.0284
N,"2,700","2,700","2,700"
State FE,No,Yes,Yes
Time FE,No,No,Yes
